# eLLM end-to-end — the LLM-net of one (N=1)

This notebook runs the **full travel loop at N=1**: the eLLM is the only LLM in
the network, with its own ewm-sm side-car, its own `hllsetLUT`, its own
codebook gate, and its own `door.py`.

```text
   discover -> broadcast -> door verdicts -> select
   [eLLM]      [network]     [every side-car]  [agent]
```

The eLLM is the *recruiter*: it sources HLLSet candidates, sends interview
invitations (broadcast), collects door verdicts, and the agent selects —
one yes → hire, several yeses → the CEO (the user) chooses. **Training teaches
the recruiter which candidates to invite first, never who to hire.**

What we run, in order:

1. the side-car and the eLLM's own codebook gate,
2. the door with the coverage guard (RoundTripGuard),
3. the broadcast seam — hire self, and a CEO-escalation preview with two doors,
4. the closed travel loop with the surprise-gated door policy v0,
5. the GPU: Qwen2.5-1.5B-Instruct + LoRA + score head,
6. the learned recruiter (`QwenBroadcastRanker`),
7. training with native ring-basis labels (soft-label KL),
8. two processes, one store: guarded publication + policy hot-swap.


In [1]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import torch

# Resolve the repo root (the notebook may be started from notebooks/ or the root).
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
    sys.path.insert(0, str(REPO_ROOT))

print("python :", sys.executable)
print("repo   :", REPO_ROOT)
print("torch  :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))

# Build the Rust side-car if the binary is missing.
BINARY = REPO_ROOT / "lattice" / "target" / "debug" / "ellm-lattice"
if not BINARY.exists():
    subprocess.run(["cargo", "build", "--offline"], cwd=REPO_ROOT / "lattice", check=True)
print("binary :", BINARY)


python : /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ellm/.venv/bin/python
repo   : /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ellm
torch  : 2.6.0+cu124 | cuda: True
gpu    : NVIDIA GeForce RTX 3060
binary : /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ellm/lattice/target/debug/ellm-lattice


In [2]:
from ellm.lattice import LatticeClient
from ellm.serve.broadcast import BroadcastDispatcher, NativeStubPolicy
from ellm.serve.surprise import SurpriseGate
from ellm.serve.server import TravelServer
from ellm.serve.generator import StubGenerator
from ellm.travel.door import Door, RoundTripGuardError
from ellm.train.trainer import Trainer, LoRALearner

print("imports ok")


imports ok


## 1. The side-car and the eLLM's own codebook gate

Every LLM in the network runs its own side-car. The gate is the eLLM's
**codebook** — the `tid{n}` encodings its door can restore. `tid{n}` is the
side-car's inscription of the host encodings, and the only encoding compatible
with the soldered lattice geometry (`P=10`, `b=32`).


In [3]:
# In-memory side-car (the demo process).
client = LatticeClient()
print("ping  :", client.ping())

tokens = [f"tid{i}" for i in range(1, 25)]
client.gate_put("ellm", tokens)
print("gates :", client.gate_list())
print("hllsetLUT entries:", len(client.hllset_lut()))


ping  : {'cmd': 'ping', 'pong': True, 'version': '0.1.0'}
gates : ['ellm']
hllsetLUT entries: 0


## 2. The door — materialize-to-prompt with the coverage guard

The door is **two-stage**: the side-car restores the `tid{n}` encodings
(stage 1, lattice-exact via `coverage = 1.0`); the host LLM would then restore
its original tokens (stage 2, inside the host). Here we watch stage 1, guarded
by the RoundTripGuard.


In [4]:
inside = client.to_prompt([1, 2, 3], user="ellm")
print("inside gate : refused =", inside["refused"], "| coverage =", inside["coverage"], "| restored =", inside["restored_ids"])

outside = client.to_prompt([1, 99], user="ellm")
print("outside gate: refused =", outside["refused"], "| coverage =", round(outside["coverage"], 3))

door = Door(client, gate="ellm")
print("door round trip [1,2,3]:", door.round_trip([1, 2, 3]))
try:
    door.open([1, 99])
except RoundTripGuardError as e:
    print("guard raised      :", e)


inside gate : refused = False | coverage = 1.0 | restored = [1, 2, 3]
outside gate: refused = True | coverage = 0.5
door round trip [1,2,3]: True
guard raised      : door refused: coverage=0.5 gate=ellm (round-trip coverage < 1.0 (RoundTripGuard))


## 3. The recruiter seam — discover → broadcast → verdicts → select

At N=1 the candidate set is the eLLM's own working set `S(t)`. The broadcast
goes to every door; the eLLM's own door answers; one yes → hire.


In [5]:
client.ingest([1, 2, 3])  # one turn, so the working set exists

dispatcher = BroadcastDispatcher(client, NativeStubPolicy(client))
candidates = dispatcher.candidates()
print("candidates at N=1:", len(candidates))

verdicts = dispatcher.broadcast(candidates[0])
print("verdicts:", [(v.user, round(v.coverage, 3), v.can_restore) for v in verdicts])

selection = dispatcher.select(verdicts)
print("selection: target =", selection.target, "| fallback =", selection.fallback,
      "| needs_ceo =", selection.needs_ceo)


candidates at N=1: 1
verdicts: [('ellm', 1.0, True)]
selection: target = ellm | fallback = False | needs_ceo = False


### CEO escalation preview (two doors, one broadcast)

This is the Phase 6 behaviour already exercised: two registered doors both
accept the same candidate → the shortlist goes to the user.


In [6]:
with LatticeClient() as fresh:
    fresh.gate_put("ellm", [f"tid{i}" for i in range(1, 9)])
    fresh.gate_put("host_b", [f"tid{i}" for i in range(1, 9)])
    fresh.ingest([1, 2])

    d = BroadcastDispatcher(fresh, NativeStubPolicy(fresh))
    verdicts = d.broadcast(d.candidates()[0])
    print("verdicts:", [(v.user, v.can_restore) for v in verdicts])

    sel = d.select(verdicts)
    print("multiple acceptors -> needs_ceo:", sel.needs_ceo, "| shortlist:", sel.needs_user)


verdicts: [('ellm', True), ('host_b', True)]
multiple acceptors -> needs_ceo: True | shortlist: ('ellm', 'host_b')


## 4. Closed travel loop — surprise-gated door policy v0

The server runs one turn: ingest → hot-swap policy → surprise-gated door →
broadcast/select → generate. The door opens when the turn commits **new** bits
(novelty spike); a no-change pass keeps the compressed lattice node.


In [7]:
server = TravelServer(client, gate="ellm", generator=StubGenerator(), surprise=SurpriseGate(theta=0.3))

first = server.turn([4, 5])
print("turn", first.turn, "| door open:", first.door.open_door, "| reason:", first.door.reason)
print("  context  :", first.context_tokens)
print("  selection:", first.selection.target, "| generated:", first.generated_ids)

second = server.turn(first.generated_ids)
print("turn", second.turn, "| door open:", second.door.open_door, "| reason:", second.door.reason)
print("  context  :", second.context_tokens)


turn 2 | door open: True | reason: surprise: novelty 0.133 >= 0.3 (residual 2.0)
  context  : ['tid4', 'tid5']
  selection: ellm | generated: [4, 5, 6]
turn 3 | door open: True | reason: surprise: novelty 0.056 >= 0.3 (residual 1.0)
  context  : ['tid4', 'tid5', 'tid6']


## 5. The GPU — Qwen2.5-1.5B-Instruct + LoRA + score head

The base model is loaded **from the local HF cache** (`local_files_only`), fp16,
on the RTX 3060. LoRA adapters are attached to the attention projections; a
small score head reads the last hidden state. This cell takes ~10s.


In [8]:
from ellm.model.base import load_qwen_hub

if not torch.cuda.is_available():
    print("CUDA not available — skipping the GPU sections.")
    hub = None
else:
    hub = load_qwen_hub()
    trainable = sum(p.numel() for p in hub.model.parameters() if p.requires_grad)
    print("hidden size:", hub.hidden_size, "| trainable LoRA+head params:", trainable)
    logits = hub.score([
        "query_bits=10 candidate_bits=5 bss=0.9000 coverage=1.0000 residual=0.0000 in_span=1 dimension=2",
        "query_bits=10 candidate_bits=7 bss=0.4000 coverage=0.7000 residual=0.1000 in_span=0 dimension=2",
    ])
    print("score batch:", [round(x, 3) for x in logits.squeeze(-1).tolist()])


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

hidden size: 1536 | trainable LoRA+head params: 2179072
score batch: [-0.213, -0.157]


## 6. The learned recruiter — `QwenBroadcastRanker`

The learned broadcast policy: for each candidate it builds a deterministic
lattice-state prompt (query bits, candidate bits, Mode B score, door coverage,
ring stats), reads the Qwen last hidden state, and scores it with the head.
The ranking order is the **invitation order** — the doors still decide the
hire.


In [9]:
if hub is not None:
    from ellm.model.ranker import QwenBroadcastRanker

    ranker = QwenBroadcastRanker(hub, client)
    state = client.state()
    query = state["working_hex"]
    ordered = ranker.discover(query, [query])
    print("ordered candidates:", len(ordered))

    feats = ranker.features(query, [query])[0]
    summary = {k: (round(v, 3) if isinstance(v, float) else v)
               for k, v in feats.items() if k not in ("query_hex", "candidate_hex")}
    print("features[0]:", summary)


ordered candidates: 1
features[0]: {'query_bits': 18, 'candidate_bits': 18, 'bss': 1.0, 'coverage': 1.0, 'residual': 0.0, 'in_span': 0, 'dimension': 0}


## 7. Training — native ring-basis labels, soft-label KL

`collect_samples` runs turns and builds `(query, candidates, native_scores)`
samples. The native label is the temperatured softmax over Mode B scores — the
Phase 5 approximation of the ring-basis soft key `w` (hard key = argmax). The
LoRA learner minimises `KL(softmax(logits/tau) || softmax(labels/tau))`.


In [10]:
if hub is not None:
    from ellm.curriculum.native_labels import collect_samples
    from ellm.model.ranker import QwenBroadcastRanker

    train_client = LatticeClient()
    train_client.gate_put("ellm", [f"tid{i}" for i in range(1, 25)])
    r2 = QwenBroadcastRanker(hub, train_client)

    samples = collect_samples(train_client, [[1, 2], [2, 3], [3, 4], [4, 5]],
                              r2, gate="ellm", tau=0.1)
    print("samples:", len(samples))
    print("native scores (sample 0):", [round(s, 3) for s in samples[0].native_scores])

    learner = LoRALearner(r2, tau=0.1)
    losses = learner.train(samples, epochs=2)
    print("losses:", [round(l, 4) for l in losses])
    print("loss decreased:", losses[-1] <= losses[0] + 1e-6)
    train_client.close()


samples: 3
native scores (sample 0): [0.966, 0.034]
losses: [0.1123, 0.4683, 0.7605, 0.0612, 0.3211, 0.53]
loss decreased: False


## 8. Two processes, one store — guarded publication + hot-swap

The trainer and the server each spawn their **own side-car over the same
persisted ewm-git store** (`ellm-lattice <repo>`). The trainer publishes the
policy only after the guard set passes; the server sees it only after
publication (commit-as-publication via the `@policy` slot).


In [11]:
import tempfile

repo = tempfile.mkdtemp(prefix="ellm-n1-")
print("shared store:", repo)

trainer_client = LatticeClient(repo_path=repo)
trainer_client.gate_put("ellm", [f"tid{i}" for i in range(1, 25)])

server_client = LatticeClient(repo_path=repo)
print("policy before training:", server_client.policy_get())

if hub is not None:
    from ellm.model.ranker import QwenBroadcastRanker

    trainer_ranker = QwenBroadcastRanker(hub, trainer_client)
    trainer = Trainer(trainer_client, schedule_every=1, ranker=trainer_ranker,
                      probes=[[1, 2, 3], [2, 3, 4]])
    published = trainer.step(turns=[[1, 2], [2, 3], [3, 4]])
    print("published:", published)
    print("server sees (hot-swap):", server_client.policy_get())


shared store: /tmp/ellm-n1-2hb2lsde
policy before training: None
published: {'generation': 1, 'theta': 0.3, 'tau': 0.1, 'lambda': 0.0, 'door': 'surprise-v0', 'probe_before': 1.0, 'probe_after': 1.0}
server sees (hot-swap): {'door': 'surprise-v0', 'generation': 1, 'lambda': 0.0, 'probe_after': 1.0, 'probe_before': 1.0, 'tau': 0.1, 'theta': 0.3}


Recovery: the server process reopens the store (its cache restores from
HEAD) and serves a turn with the **real Qwen generator** inside the
materialized context.


In [12]:
server_client.close()
server_client = LatticeClient(repo_path=repo)  # recovery: StateCache::restore from HEAD

from ellm.model.generator_qwen import QwenGenerator
generator = QwenGenerator(hub) if hub is not None else StubGenerator()

server = TravelServer(server_client, gate="ellm", generator=generator)
result = server.turn([7, 8])
print("policy generation:", (result.policy or {}).get("generation"))
print("door open:", result.door.open_door, "| selection:", result.selection.target)
print("context:", result.context_tokens)
print("generated:", result.generated_ids)

trainer_client.close()
server_client.close()


policy generation: 1
door open: True | selection: ellm
context: ['tid7', 'tid8']
generated: [7, 8, 9, 10, 11, 12, 13, 1]


## What Phase 6 adds

At N=1 the broadcast always hears one door — the eLLM's own. Phase 6 stands up
a **second LLM** (its own ewm-sm, `hllsetLUT`, `H(t)`, `door.py`) and shows the
broadcast collecting **two door verdicts with no change to the eLLM**: no
retraining, no registry update, no architecture switch. The CEO-escalation
cell above is exactly that machinery, already running.
